In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [12]:
%cd "/content/drive/MyDrive/Sign-Language-To-Text-Conversion"
!ls


/content/drive/MyDrive/Sign-Language-To-Text-Conversion
 Application.py
 asl_cnn_myversion.h5
'Bachelor'\''s-Project Report-(Sign Language To Text Conversion).pdf'
 dataSet
 FoldersCreation.py
 images
 LICENSE
 Models
'Photo Booth Library.zip'
'Photo on 12-02-26 at 23.40.jpg'
'Photo on 12-02-26 at 23.42 (1).jpg'
'Photo on 12-02-26 at 23.42.jpg'
'PROJECT PRESENTATION - SIGN LANGUAGE TO TEXT CONVERSION.pptx'
 README.md
'safeimagekit-resized-WhatsApp Image 2026-02-12 at 23.49.23 (1).png'
'safeimagekit-resized-WhatsApp Image 2026-02-12 at 23.49.23.png'
'safeimagekit-resized-WhatsApp Image 2026-02-12 at 23.55.42.png'
 TestingDataCollection.py
 TrainingDataCollection.py
'WhatsApp Image 2026-02-12 at 23.49.23.jpeg'


In [13]:
import os
import cv2
import numpy as np

IMG_SIZE = 128

# Map classes: folder names used in dataSet
# In original repo: 0, A, B, ..., Z  (0 = blank)
classes = ['0'] + [chr(ord('A') + i) for i in range(26)]
class_to_idx = {c: i for i, c in enumerate(classes)}

def preprocess_image(path, size=IMG_SIZE):
    img = cv2.imread(path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 2)
    th3 = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11, 2
    )
    _, binary = cv2.threshold(
        th3, 70, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    binary = cv2.resize(binary, (size, size))
    binary = binary.astype('float32') / 255.0
    binary = np.expand_dims(binary, axis=-1)
    return binary

def load_dataset(split='trainingData'):
    X, y = [], []
    base_dir = os.path.join('dataSet', split)
    for cls in classes:
        folder = os.path.join(base_dir, cls)
        if not os.path.isdir(folder):
            continue
        label = class_to_idx[cls]
        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            img = preprocess_image(fpath)
            if img is None:
                continue
            X.append(img)
            y.append(label)
    X = np.array(X)
    y = np.array(y)
    return X, y

X_train, y_train = load_dataset('trainingData')
X_test, y_test = load_dataset('testingData')

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(12853, 128, 128, 1) (12853,)
(4268, 128, 128, 1) (4268,)


In [16]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os
import cv2

IMG_SIZE = 128

classes = ['0'] + [chr(ord('A') + i) for i in range(26)]

dataset_path = "/content/drive/MyDrive/Sign-Language-To-Text-Conversion/dataSet/testingData" # Changed from "dataset" to "dataSet"

X = []
y = []

for label in classes:

    folder = os.path.join(dataset_path, label)

    for img_name in os.listdir(folder):

        path = os.path.join(folder, img_name)

        img = cv2.imread(path)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        img = cv2.resize(gray,(IMG_SIZE,IMG_SIZE))

        X.append(img)
        y.append(classes.index(label))


X = np.array(X)/255.0
X = X.reshape(-1,IMG_SIZE,IMG_SIZE,1)

y = tf.keras.utils.to_categorical(y, len(classes))


model = models.Sequential([

layers.Conv2D(32,(3,3),activation='relu',input_shape=(128,128,1)),
layers.MaxPooling2D((2,2)),

layers.Conv2D(64,(3,3),activation='relu'),
layers.MaxPooling2D((2,2)),

layers.Conv2D(128,(3,3),activation='relu'),
layers.MaxPooling2D((2,2)),

layers.Flatten(),

layers.Dense(128,activation='relu'),

layers.Dropout(0.5),

layers.Dense(len(classes),activation='softmax')

])


model.compile(
optimizer='adam',
loss='categorical_crossentropy',
metrics=['accuracy']
)


history = model.fit(
X,
y,
epochs=10,
validation_split=0.2
)


model.save("/content/drive/MyDrive/Sign-Language-To-Text-Conversion/Models/sign_model.keras")

print("Model Saved Successfully")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 15s 86ms/step - accuracy: 0.3285 - loss: 2.2810 - val_accuracy: 0.0375 - val_loss: 11.4858
Epoch 2/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.8883 - loss: 0.3239 - val_accuracy: 0.0679 - val_loss: 13.6484
Epoch 3/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9570 - loss: 0.1428 - val_accuracy: 0.0796 - val_loss: 16.3130
Epoch 4/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9620 - loss: 0.1065 - val_accuracy: 0.0796 - val_loss: 18.9830
Epoch 5/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9682 - loss: 0.0898 - val_accuracy: 0.0796 - val_loss: 21.5152
Epoch 6/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.9742 - loss: 0.0749 - val_accuracy: 0.0796 - val_loss: 23.8971
Epoch 7/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.9802 - loss: 0.0558 - val_accuracy: 0.0796 - val_loss: 21.7310
Epoch 8/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.9839 - loss: 0.0485 - 